ШАГ 6: КЛАССИФИКАЦИЯ (SI > МЕДИАНЫ)

Целью данного этапа является построение и сравнение моделей бинарной классификации для прогнозирования превышения индексом селективности медианного значения выборки.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('cleaned_data.csv')

targets_to_exclude = ['IC50, mM', 'CC50, mM', 'SI', 'pIC50', 'pCC50', 'log_SI',
                      'IC50_above_med', 'CC50_above_med', 'SI_above_med', 'SI_above_8']
X = df.drop(columns=targets_to_exclude)
y = df['SI_above_med']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}\n")

pipelines_clf = {
    'LogisticRegression': Pipeline([('scaler', StandardScaler()),
                                    ('model', LogisticRegression(random_state=42, max_iter=1000))]),
    'SVC': Pipeline([('scaler', StandardScaler()),
                     ('model', SVC(random_state=42))]),
    'RandomForest': Pipeline([('scaler', StandardScaler()),
                              ('model', RandomForestClassifier(random_state=42))])
}

param_grids_clf = {
    'LogisticRegression': {
        'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'model__penalty': ['l2']
    },
    'SVC': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear']
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5]
    }
}

results = []

for name in pipelines_clf:
    grid = GridSearchCV(pipelines_clf[name], param_grids_clf[name], cv=5,
                        scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    y_pred = grid.best_estimator_.predict(X_test)

    results.append({
        'модель': name,
        'лучшие параметры': str(grid.best_params_),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
print("результаты классификации:")
print(results_df.to_string(index=False))

Данные загружены
Train: X=(800, 139), y=(800,)
Test:  X=(201, 139), y=(201,)

Обучение LogisticRegression...
Обучение SVC...
Обучение RandomForest...

Результаты классификации (SI_above_med):
            Модель                                                                      Лучшие параметры  Accuracy  Precision  Recall  F1 Score
               SVC                                               {'model__C': 1, 'model__kernel': 'rbf'}  0.701493   0.750000    0.60  0.666667
      RandomForest {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 200}  0.661692   0.686047    0.59  0.634409
LogisticRegression                                             {'model__C': 0.1, 'model__penalty': 'l2'}  0.557214   0.557895    0.53  0.543590


ВЫВОДЫ: 

Наилучшие результаты продемонстрировал метод опорных векторов (SVC) с нелинейным RBF-ядром (Accuracy ≈ 0.70), который позволил эффективнее разделять классы в многомерном пространстве признаков. Случайный лес уступил SVC (Accuracy ≈ 0.66); я полагаю, что недостаточная глубина деревьев или ограниченное число эстиматоров не позволили полностью уловить сложные зависимости, а возможный шум в данных мог провоцировать переобучение отдельных деревьев. Логистическая регрессия показала результат, близкий к случайному угадыванию (Accuracy ≈ 0.55), что подтверждает нелинейный характер связей в исходных данных и слабую применимость линейных методов к данной задаче.

Общее качество предсказания целевой метрики оказалось невысоким. По моему мнению, это обусловлено тем, что целевая переменная является производной от других показателей, и её прямое прогнозирование классическими алгоритмами затруднено накоплением дисперсии признаков и вероятным зашумлением исходной разметки.

Для улучшения качества классификации я рекомендую обратиться к более мощным ансамблевым методам, таким как градиентный бустинг (XGBoost, LightGBM, CatBoost), или нейросетевым архитектурам. Целесообразной представляется дополнительная математическая проработка признаков: применение метода главных компонент (PCA) или процедур отбора информативных дескрипторов для снижения размерности и уменьшения шума. Сбор дополнительных данных также может способствовать более стабильному обучению сложных алгоритмов. Альтернативным подходом я вижу раздельное предсказание базовых показателей с помощью регрессионных моделей и последующее вычисление итогового класса на основе их прогнозов.

